# Manhattan Starbucks: Interactive Strategy Map

An **interactive Folium map** integrating all key findings from the Starbucks spatial analysis series into a single, explorable visualization.

**Layers:**
- All 171 Starbucks locations colored by Location Fitness Score
- 1,200+ competitor cafes
- Subway stations sized by ridership
- Expansion opportunities, closure risks, and competitive white space

Toggle layers on/off using the control panel in the top-right corner.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    _candidates = [
        Path("/kaggle/input/manhattan-cafe-wars"),
        Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
    ]
    DATA_DIR = next((p for p in _candidates if p.exists()), None)
    if DATA_DIR is None:
        raise FileNotFoundError("Dataset not found")
else:
    DATA_DIR = Path("../../dataset-upload/v3")

stores = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")
cafes = pd.read_csv(DATA_DIR / "manhattan_cafes_osm.csv")
mta = pd.read_csv(DATA_DIR / "manhattan_mta_ridership_summary.csv")

print(f"Starbucks: {len(stores)} | Competitors: {len(cafes)} | MTA stations: {len(mta)}")

## Compute Scores

In [ ]:
def norm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df = stores.dropna(subset=["lat", "lon", "location_fitness_score",
                            "mta_avg_daily_ridership", "n_starbucks_500m"]).copy()

# Expansion score
df["expansion_score"] = (
    norm(df["mta_avg_daily_ridership"]) * 0.3 +
    norm(df["ped_count_nearest"].fillna(0)) * 0.2 +
    norm(df["nearest_starbucks_dist_m"]) * 0.25 +
    (1 - norm(df["n_starbucks_500m"])) * 0.25
)

# Closure risk
df["closure_risk"] = (
    (1 - norm(df["location_fitness_score"])) * 0.4 +
    norm(df["n_starbucks_500m"]) * 0.3 +
    (1 - norm(df["mta_avg_daily_ridership"])) * 0.3
)

top_expand = df.nlargest(5, "expansion_score")
top_risk = df.nlargest(5, "closure_risk")

print(f"Scored {len(df)} stores")

## Build Interactive Map

In [ ]:
# ── base map ──────────────────────────────────────────────────────────
m = folium.Map(location=[40.758, -73.985], zoom_start=13,
               tiles="CartoDB positron")

# ── Layer 1: All Starbucks by LFS ────────────────────────────────────
lfs_layer = folium.FeatureGroup(name="Starbucks (by LFS)", show=True)

def lfs_color(score):
    if score > df["location_fitness_score"].quantile(0.67):
        return "#00704A"  # green — strong
    elif score > df["location_fitness_score"].quantile(0.33):
        return "#FFA726"  # orange — medium
    else:
        return "#D32F2F"  # red — weak

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=lfs_color(row["location_fitness_score"]),
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Starbucks</b><br>"
            f"LFS: {row['location_fitness_score']:.3f}<br>"
            f"Ridership: {row['mta_avg_daily_ridership']:,.0f}<br>"
            f"Starbucks 500m: {row['n_starbucks_500m']:.0f}",
            max_width=200
        ),
    ).add_to(lfs_layer)

lfs_layer.add_to(m)

# ── Layer 2: Competitors (clustered) ─────────────────────────────────
comp_layer = folium.FeatureGroup(name="Competitors (1,200+)", show=False)
comp_cluster = MarkerCluster()

for _, row in cafes.dropna(subset=["lat", "lon"]).iterrows():
    name = row.get("name", "Cafe")
    if pd.isna(name):
        name = "Cafe"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="#9E9E9E",
        fill=True,
        fill_opacity=0.5,
        popup=str(name)[:50],
    ).add_to(comp_cluster)

comp_cluster.add_to(comp_layer)
comp_layer.add_to(m)

# ── Layer 3: MTA Stations ────────────────────────────────────────────
mta_layer = folium.FeatureGroup(name="Subway Stations", show=False)

max_rider = mta["avg_daily_ridership"].max()
for _, row in mta.dropna(subset=["lat", "lon"]).iterrows():
    r = 3 + (row["avg_daily_ridership"] / max_rider) * 12
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=r,
        color="#1976D2",
        fill=True,
        fill_opacity=0.4,
        popup=folium.Popup(
            f"<b>{row['station_name'][:40]}</b><br>"
            f"Daily ridership: {row['avg_daily_ridership']:,.0f}",
            max_width=250
        ),
    ).add_to(mta_layer)

mta_layer.add_to(m)

# ── Layer 4: Expansion Opportunities ─────────────────────────────────
expand_layer = folium.FeatureGroup(name="★ Expansion Opportunities", show=True)

for rank, (_, row) in enumerate(top_expand.iterrows(), 1):
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.Icon(color="green", icon="star", prefix="fa"),
        popup=folium.Popup(
            f"<b>Expansion #{rank}</b><br>"
            f"Score: {row['expansion_score']:.3f}<br>"
            f"Ridership: {row['mta_avg_daily_ridership']:,.0f}<br>"
            f"Nearest SB: {row['nearest_starbucks_dist_m']:.0f}m",
            max_width=200
        ),
    ).add_to(expand_layer)

expand_layer.add_to(m)

# ── Layer 5: Closure Risk ────────────────────────────────────────────
risk_layer = folium.FeatureGroup(name="✕ Closure Risk", show=True)

for rank, (_, row) in enumerate(top_risk.iterrows(), 1):
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.Icon(color="red", icon="warning", prefix="fa"),
        popup=folium.Popup(
            f"<b>Closure Risk #{rank}</b><br>"
            f"LFS: {row['location_fitness_score']:.3f}<br>"
            f"Risk score: {row['closure_risk']:.3f}<br>"
            f"Starbucks 500m: {row['n_starbucks_500m']:.0f}",
            max_width=200
        ),
    ).add_to(risk_layer)

risk_layer.add_to(m)

# ── layer control ────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m)

print("Map built with 5 layers. Toggle layers using the panel on the right.")
m

## How to Read This Map

| Layer | What it shows |
|-------|---------------|
| **Starbucks (by LFS)** | Green = strong fit, Orange = medium, Red = weak |
| **Competitors** | 1,200+ cafes clustered. Zoom in to see individual locations |
| **Subway Stations** | Blue circles sized by daily ridership |
| **★ Expansion** | Green stars = top 5 underserved high-demand locations |
| **✕ Closure Risk** | Red markers = top 5 oversaturated low-demand stores |

**Key insight:** Toggle on Subway Stations + Starbucks together — you'll see that green (strong) stores cluster around large blue circles (high ridership), confirming our r=0.58 correlation finding.

## Series Navigation

| # | Notebook | Link |
|---|----------|------|
| 0 | Manhattan Café Wars — EDA | [Kaggle](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| 1 | Starbucks 10-K NLP: Topic & Keyword Analysis | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| 1F | Starbucks 10-K LDA Topic Explorer (pyLDAvis) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) |
| 2A | Starbucks Spatial Clustering | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| 2B | Starbucks Location Fitness | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| 2C | Starbucks Walk-Distance Analysis (OSMnx) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) |
| 5A | LFS Predictive Model | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness-score-predictive-model) |
| 5B | Strategic Location Insights | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-strategic-location-insights) |
| 5C | Optimal Store Placement | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-optimal-store-placement) |
| 5D | Hourly Demand Patterns | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-hourly-demand-patterns) |
| **5E** | **Interactive Strategy Map (this notebook)** | — |
| C | Data Pipeline: EDGAR + OSM → CSV | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |
| D | US Expansion Animated Choropleth | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) |
| G | NLP × Spatial Integration | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-nlp-x-spatial) |

**Dataset:** [shiratoriseto/manhattan-cafe-wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars)  
**GitHub:** [seto-siratori/starbucks-kaggle](https://github.com/seto-siratori/starbucks-kaggle)